In [2]:
import pandas as pd 
import re 

def aggregate_associations(df):
    # Dictionary to hold cue -> association counts
    associations_dict = {}
    
    # Iterate over each row in the DataFrame
    for index, row in df.iterrows():
        cue = row['cue']
        # Columns that represent associations
        association_columns = [col for col in df.columns if 'R' in col and 'Raw' not in col]

        # Update the counts for each association
        for col in association_columns:
            association = row[col]
            if pd.notna(association):
                if cue not in associations_dict:
                    associations_dict[cue] = {}
                if association in associations_dict[cue]:
                    associations_dict[cue][association] += 1
                else:
                    associations_dict[cue][association] = 1

    return associations_dict


def preprocess_associations(association_counts):
    data_for_excel = []
    # Regular expression to match hexadecimal hash patterns
    hex_pattern = re.compile(r'^[a-f0-9]{32}$', re.IGNORECASE)
    
    # Keep track of hex patterns we find
    hex_found = []

    # Filter and transform the data
    for cue, associations in association_counts.items():
        for association, freq in associations.items():
            # Clean up association value to remove leading/trailing whitespaces
            association = association.strip()
            
            # Debug: Check if this looks like a hex but isn't matching
            if any(c in association.lower() for c in '0123456789abcdef') and len(association) >= 30:
                print("Cue: ", cue)
                print(f"Potential hex value found: '{association}'")
                print(f"Length: {len(association)}")
                print(f"Matches pattern: {bool(hex_pattern.match(association))}")
                print("ASCII values:", [ord(c) for c in association])
                print("---")

            # Exclude associations if they match the hexadecimal pattern, are '#Missing', '#Unknown', or the same as 'cue'
            if association not in ['#Missing', '#Unknown'] and association != cue and not hex_pattern.match(association):
                data_for_excel.append({"relation": "forwardassociated", "cue": cue, "association": association, "freq": freq})
            elif hex_pattern.match(association):
                hex_found.append(association)
    
    print(f"\nTotal hex patterns found: {len(hex_found)}")
    if hex_found:
        print("Sample of hex patterns found:", hex_found[:5])
        
    return data_for_excel

def calculate_statistics(data_for_excel):
    # Convert to DataFrame
    df = pd.DataFrame(data_for_excel)

    # Total number of unique cues
    num_cues = df['cue'].nunique()
    
    # Average number of association types per cue
    avg_associations_per_cue = df.groupby('cue').size().mean()
    
    # Total number of associations considering their frequency (weighted sum)
    total_associations_weighted = df['freq'].sum()
    
    # Total number of unique associations (distinct association types)
    total_unique_associations = df['association'].nunique()

    # Print the statistics
    print(f"Total number of cues: {num_cues}")
    print(f"Average number of association types per cue: {avg_associations_per_cue:.2f}")
    print(f"Total number of associations (weighted by frequency): {total_associations_weighted}")
    print(f"Total number of unique association types: {total_unique_associations}")
                                                       
def export_to_excel(data_for_excel, filename):
    # Convert to DataFrame
    df_to_excel = pd.DataFrame(data_for_excel)
    # Sort by 'cue' and 'freq' if needed
    df_to_excel.sort_values(by=['cue', 'freq'], ascending=[True, False], inplace=True)
    # Write to Excel
    df_to_excel.to_excel(filename, index=False)
    print(f"Data exported to {filename}")

# Call the function
# path = "../data/SWOW/SWOW-ZH24/SWOWZH.R55.20230424.csv"

path = "./SWOW-EN.R100.20180827.csv"
out_filename = "./SWOW-US.R100.20180827.processed.xlsx"
df = pd.read_csv(path)
dfus = df.query("country=='United States' and nativeLanguage=='United States'")

association_counts = aggregate_associations(dfus)

processed_data = preprocess_associations(association_counts)
calculate_statistics(processed_data)

export_to_excel(processed_data, out_filename)

Cue:  divide
Potential hex value found: 'Sundays in the park with George'
Length: 31
Matches pattern: False
ASCII values: [83, 117, 110, 100, 97, 121, 115, 32, 105, 110, 32, 116, 104, 101, 32, 112, 97, 114, 107, 32, 119, 105, 116, 104, 32, 71, 101, 111, 114, 103, 101]
---
Cue:  type
Potential hex value found: 'to create text with a typewriter or computer'
Length: 44
Matches pattern: False
ASCII values: [116, 111, 32, 99, 114, 101, 97, 116, 101, 32, 116, 101, 120, 116, 32, 119, 105, 116, 104, 32, 97, 32, 116, 121, 112, 101, 119, 114, 105, 116, 101, 114, 32, 111, 114, 32, 99, 111, 109, 112, 117, 116, 101, 114]
---
Cue:  hold
Potential hex value found: 'place a request that an item be reserved for you'
Length: 48
Matches pattern: False
ASCII values: [112, 108, 97, 99, 101, 32, 97, 32, 114, 101, 113, 117, 101, 115, 116, 32, 116, 104, 97, 116, 32, 97, 110, 32, 105, 116, 101, 109, 32, 98, 101, 32, 114, 101, 115, 101, 114, 118, 101, 100, 32, 102, 111, 114, 32, 121, 111, 117]
---
Cue:  meaning

In [3]:
dfus.head()

,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
656,657,988,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,divide,conquer,apart,split
657,658,989,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,describe,tell,paper,words
658,659,990,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,expense,report,money,Mad Men
659,660,991,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,worth,money,expensive,value
660,661,992,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,be,is,are,am


In [21]:
# check if R3 has NA values
dfus['R3'].isna().sum()

70152

In [10]:
# find the first index where R3 contains na value
dfus['R3'].isna().idxmax()

739

In [11]:
dfus.loc[739]

Unnamed: 0                        740
id                               1086
participantID                     104
age                                34
gender                             Fe
nativeLanguage          United States
country                 United States
education                         NaN
created_at        2011-08-15 07:30:39
cue                              upon
R1                               once
R2                          fairytale
R3                                NaN
Name: 739, dtype: object

In [12]:
dfus['R2'].isna().idxmax()

813

In [13]:
dfus.loc[813]

Unnamed: 0                        814
id                               1244
participantID                     121
age                                47
gender                             Fe
nativeLanguage          United States
country                 United States
education                         NaN
created_at        2011-08-15 09:03:33
cue                             allow
R1                             permit
R2                                NaN
R3                                NaN
Name: 813, dtype: object

In [14]:
dfus['R1'].isna().idxmax() 

1104

In [15]:
dfus.loc[1104]

Unnamed: 0                       1105
id                               1822
participantID                     175
age                                21
gender                             Fe
nativeLanguage          United States
country                 United States
education                         NaN
created_at        2011-08-15 14:38:55
cue                              poor
R1                                NaN
R2                                NaN
R3                                NaN
Name: 1104, dtype: object

In [16]:
from tqdm.auto import tqdm

In [17]:
# need to assert that if current R has a value, then front R must not be NA
# e.g., if R2 is not NA, then R1 must not be NA
issue_row_ids = []
for idx, row in tqdm(dfus.iterrows(), total=dfus.shape[0]):
    if pd.notna(row['R2']) and pd.isna(row['R1']):
        issue_row_ids.append(idx)
    if pd.notna(row['R3']) and pd.isna(row['R2']):
        issue_row_ids.append(idx)

  0%|          | 0/604342 [00:00<?, ?it/s]

In [22]:
issue_row_ids = list(set(issue_row_ids))
print(len(issue_row_ids))

50


In [23]:
# remove those ids 

cleaned_dfus = dfus[~dfus.index.isin(issue_row_ids)]

In [24]:
print(len(cleaned_dfus))
print(len(dfus))

604292
604342


In [ ]:
cleaned_dfus.iloc[0]

Unnamed: 0                        657
id                                988
participantID                      94
age                                27
gender                             Fe
nativeLanguage          United States
country                 United States
education                         NaN
created_at        2011-08-15 07:01:48
cue                            divide
R1                            conquer
R2                              apart
R3                              split
Name: 656, dtype: object

In [27]:
pd.isna(cleaned_dfus.iloc[0]['education'])

True

In [28]:
SYSTEM_PROMPT = """You are a human participant in a psychology experiment."""
INSTRUCTION_PROMPT = """### Background ###
On average, an adult knows about 40,000 words, but what do these words mean to people like you and me? You can help scientists understand how meaning is organized in our mental dictionary by playing the game of word associations. This game is easy: Just give the first three words that come to mind for a given cue word.
### OUTPUT FORMAT ###
Output your response in the following format:
response1, response2, response3
Do not provide any additional context, or explanations. Just the words as comma-separated values.
If you cannot think of further responses after response1 or response2, output the token NO MORE RESPONSE for the remaining slot(s).
### End of instructions ###
"""

INPUT_PROMPT="""Cue word: {keyword}"""

def form_alpaca_format_data(row):
    if pd.isna(row['cue']):
        return None
    output_content = ""
    r1 = row['R1']
    r2 = row['R2']
    r3 = row['R3']
    if pd.isna(r2):
        r2 = "NO MORE RESPONSE"
    if pd.isna(r3):
        r3 = "NO MORE RESPONSE"
    output_content = f"{r1}, {r2}, {r3}"
    data = {
        "system": SYSTEM_PROMPT,
        "instruction": INSTRUCTION_PROMPT,
        "input": INPUT_PROMPT.format(keyword=row['cue']),
        "output": output_content
    }
    return data

In [29]:
alpaca_data = df.apply(form_alpaca_format_data, axis=1).dropna().tolist()

In [32]:
alpaca_data[0]

{'system': 'You are a human participant in a psychology experiment.',
 'instruction': '### Background ###\nOn average, an adult knows about 40,000 words, but what do these words mean to people like you and me? You can help scientists understand how meaning is organized in our mental dictionary by playing the game of word associations. This game is easy: Just give the first three words that come to mind for a given cue word.\n### OUTPUT FORMAT ###\nOutput your response in the following format:\nresponse1, response2, response3\nDo not provide any additional context, or explanations. Just the words as comma-separated values.\nIf you cannot think of further responses after response1 or response2, output the token NO MORE RESPONSE for the remaining slot(s).\n### End of instructions ###\n',
 'input': 'Cue word: although',
 'output': 'nevertheless, yet, but'}

In [30]:
# install jsonlines
!pip install jsonlines

In [33]:
# shuffle alpaca_data
import random 

random.shuffle(alpaca_data)


In [35]:
import jsonlines

with jsonlines.open('alpaca_data_swow_us.jsonl', mode='w') as writer:
    writer.write_all(alpaca_data)


In [19]:
issue_row_ids

[151041,
 565381,
 602758,
 133000,
 195721,
 22154,
 195723,
 589577,
 195725,
 1209999,
 191506,
 336147,
 321684,
 33688,
 335897,
 1206948,
 288381,
 383607,
 516393,
 274220,
 224561,
 244405,
 280378,
 243387,
 234814,
 347511,
 152516,
 169029,
 275399,
 387277,
 400591,
 636368,
 691154,
 325843,
 164827,
 353503,
 195424,
 163171,
 449766,
 137449,
 132717,
 453487,
 172530,
 518388,
 633846,
 50679,
 116089,
 70010,
 47229,
 402942]

In [20]:
dfus.loc[195721]

Unnamed: 0                     195722
id                             237674
participantID                   20835
age                                17
gender                             Ma
nativeLanguage          United States
country                 United States
education                         NaN
created_at        2012-02-10 16:54:10
cue                         temporary
R1                                NaN
R2                              files
R3                                NaN
Name: 195721, dtype: object

In [ ]:
# check unique values in R3 with ordering, descending
dfus['R3'].value_counts()

R3
water           1852
money           1695
food            1633
green           1207
work            1184
                ... 
war paint          1
burgundy           1
Nate Freeman       1
antigone           1
in a lie           1
Name: count, Length: 52558, dtype: int64

In [6]:
dfus['R3'].str.contains('NO MORE RESPONSE').sum()

0

In [37]:
dfus.head()
dfus['nativeLanguage'].value_counts()

nativeLanguage
United States    604342
Name: count, dtype: int64

In [ ]:
dfus1 = pd.read_excel("./SWOW-EN18/SWOW-US.R100.20180827.processed.xlsx")
dfus1.head()

,relation,cue,association,freq
0,forwardassociated,Abel,Cain,31
1,forwardassociated,Abel,Bible,14
2,forwardassociated,Abel,murder,4
3,forwardassociated,Abel,brother,4
4,forwardassociated,Abel,Adam,3


In [47]:
num_cues = len(set(dfus1['cue']))
num_association = len(set(dfus1['association']))

print(num_cues, num_association)
print(len(dfus1.index))

12282 88254
819223


In [53]:
dfus1.tail(50)

,relation,cue,association,freq
819173,forwardassociated,zucchini,thin,1
819174,forwardassociated,NaN,void,36
819175,forwardassociated,NaN,zero,19
819176,forwardassociated,NaN,nothing,17
819177,forwardassociated,NaN,empty,10
819178,forwardassociated,NaN,none,7
819179,forwardassociated,NaN,set,4
819180,forwardassociated,NaN,nil,3
819181,forwardassociated,NaN,invalid,3
819182,forwardassociated,NaN,hypothesis,3


In [32]:
# display(df1us.head())
df1us['nativeLanguage'].value_counts()

nativeLanguage
United States          604342
Canada                  34989
United Kingdom           7121
Other_Foreign            3999
Spanish                  3351
Australia                2496
Russian                  1491
Mandarin                 1407
German                   1224
French                    840
India                     735
Other_English             690
Italian                   624
Ireland                   594
Portuguese                565
Dutch_Netherlands         517
Polish                    489
Hindi                     464
New Zealand               462
Korean                    454
Bengali                   399
Singapore                 375
Tamil                     320
South Africa              290
Hebrew                    280
Vietnamese                263
Turkish                   261
Puerto Rico               241
Dutch_Flanders            232
Arabic                    213
Danish                    207
Malay_Indonesian          182
Persian                  

In [28]:
df1['country'].value_counts()

country
United States                        671521
United Kingdom                       161610
Canada                                94072
Australia                             72749
Germany                               17401
Belgium                               13072
New Zealand                           11904
Netherlands                           11567
France                                 9447
Spain                                  8825
Ireland                                7548
Finland                                7397
India                                  7368
notfound                               6403
Sweden                                 6261
South Africa                           6227
99                                     5363
Italy                                  5341
Denmark                                5189
Singapore                              4156
Norway                                 4146
Israel                                 4041
Switzerland             

In [11]:
len(df1.index)

1228200

In [ ]:


df1us.head()

,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
656,657,988,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,divide,conquer,apart,split
657,658,989,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,describe,tell,paper,words
658,659,990,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,expense,report,money,Mad Men
659,660,991,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,worth,money,expensive,value
660,661,992,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,be,is,are,am


In [10]:
len(df1us.index)

671521

In [ ]:
path = "../../MDAP-2025/Project/Data/SWOW/SWOWEN.spellchecked.27-06-2022.csv"
df2 = pd.read_csv(path)
df2.head()

,id,participantID,age,gender,nativeLanguage,education,city,region,country,section,cue,RT1,RT2,RT3,created_at,R1,R2,R3
0,29.0,3.0,33,Fe,United States,0,Adelaide,South Australia,Australia,seed,although,5077.0,9751.0,510.0,2011-08-12 02:19:38,nevertheless,yet,but
1,30.0,3.0,33,Fe,United States,0,Adelaide,South Australia,Australia,seed,deal,2354.0,693.0,2187.0,2011-08-12 02:19:38,no,cards,shake
2,31.0,3.0,33,Fe,United States,0,Adelaide,South Australia,Australia,seed,music,1785.0,706.0,1626.0,2011-08-12 02:19:38,notes,band,rhythm
3,32.0,3.0,33,Fe,United States,0,Adelaide,South Australia,Australia,seed,inform,1323.0,738.0,1107.0,2011-08-12 02:19:38,tell,rat on,NaN
4,33.0,3.0,33,Fe,United States,0,Adelaide,South Australia,Australia,seed,way,1914.0,618.0,930.0,2011-08-12 02:19:38,path,via,method


In [18]:
print(len(df1.index))

pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_rows', 1000)
display(df1['country'].value_counts())  


1228200


country
United States                        671521
United Kingdom                       161610
Canada                                94072
Australia                             72749
Germany                               17401
Belgium                               13072
New Zealand                           11904
Netherlands                           11567
France                                 9447
Spain                                  8825
Ireland                                7548
Finland                                7397
India                                  7368
notfound                               6403
Sweden                                 6261
South Africa                           6227
99                                     5363
Italy                                  5341
Denmark                                5189
Singapore                              4156
Norway                                 4146
Israel                                 4041
Switzerland             

In [ ]:
print(len(df2.index))
pd.set_option('display.max_colwidth', 1000)
pd.set_option('display.max_rows', 1000)
display(df2['country'].value_counts())  


1749335


country
United States                        905473
United Kingdom                       217812
Canada                               122983
Australia                            100586
Germany                               28738
Belgium                               21969
Netherlands                           20676
India                                 18822
New Zealand                           16103
Spain                                 13891
France                                13888
Singapore                             13091
Finland                               11195
Italy                                 11153
Sweden                                11079
Ireland                               10800
South Africa                           8189
Denmark                                7744
notfound                               7261
99                                     7056
Norway                                 6851
Philippines                            6850
Israel                  

In [6]:

df2us = df1.query("country=='United States'")
df2us.head()

,Unnamed: 0,id,participantID,age,gender,nativeLanguage,country,education,created_at,cue,R1,R2,R3
656,657,988,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,divide,conquer,apart,split
657,658,989,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,describe,tell,paper,words
658,659,990,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,expense,report,money,Mad Men
659,660,991,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,worth,money,expensive,value
660,661,992,94,27,Fe,United States,United States,NaN,2011-08-15 07:01:48,be,is,are,am


In [19]:
df2us['created_at']

656        2011-08-15 07:01:48
657        2011-08-15 07:01:48
658        2011-08-15 07:01:48
659        2011-08-15 07:01:48
660        2011-08-15 07:01:48
                  ...         
1228123    2018-01-08 18:04:37
1228124    2018-01-08 18:04:37
1228125    2018-01-08 18:04:37
1228126    2018-01-08 18:04:37
1228127    2018-01-08 18:04:37
Name: created_at, Length: 671521, dtype: object

In [20]:
df1us['created_at']

656        2011-08-15 07:01:48
657        2011-08-15 07:01:48
658        2011-08-15 07:01:48
659        2011-08-15 07:01:48
660        2011-08-15 07:01:48
                  ...         
1228123    2018-01-08 18:04:37
1228124    2018-01-08 18:04:37
1228125    2018-01-08 18:04:37
1228126    2018-01-08 18:04:37
1228127    2018-01-08 18:04:37
Name: created_at, Length: 671521, dtype: object

In [8]:
len(df2us.index)

671521